In [ ]:
# Colab Setup: Run this cell first!
%pip install -q openai pydantic

from google.colab import userdata, drive
import os

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

# Mount Google Drive for saving output files
drive.mount("/content/drive")

# 05 - Putting It All Together: Player Cards

This is the payoff. We're going to combine everything you've learned:

- **Functions** — Reusable code
- **Loops** — Process multiple texts (SCALE)
- **API calls** — Talk to AI from code (CHAINING)
- **Structured output** — Get organized data (CONTROL)

**Goal:** Loop through 3 passages, extract structured data, generate card art with DALL-E, and save everything for use in a Next.js app.

---

## Setup

In [ ]:
import os
import json
import requests
from pathlib import Path
from pydantic import BaseModel
from IPython.display import Image, display

if not os.environ.get("OPENAI_API_KEY"):
    raise EnvironmentError("OPENAI_API_KEY not found! Add it to Colab Secrets.")

from openai import OpenAI
client = OpenAI()

# Output paths (Google Drive)
IMAGES_DIR = Path("/content/drive/MyDrive/complit-126x/cards")
DATA_DIR = Path("/content/drive/MyDrive/complit-126x/data")

# Create directories if they don't exist
IMAGES_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

print("✓ Ready")
print(f"✓ Images will be saved to: {IMAGES_DIR}")
print(f"✓ Data will be saved to: {DATA_DIR}")

## Our Three Passages

Three creation narratives from different traditions:

In [ ]:
passages = [
    {
        "id": "genesis",
        "name": "Genesis",
        "text": """In the beginning God created the heaven and the earth. And the earth was without form, and void; and darkness was upon the face of the deep. And the Spirit of God moved upon the face of the waters. And God said, Let there be light: and there was light. And God saw the light, that it was good: and God divided the light from the darkness."""
    },
    {
        "id": "theogony",
        "name": "Hesiod's Theogony",
        "text": """First of all, the Void came into being, next broad-bosomed Earth, the solid and eternal home of all, and Eros, the most beautiful of the immortal gods, who in every man and every god softens the sinews and overpowers the prudent purpose of the mind. Out of Void came Darkness and black Night, and out of Night came Light and Day, her children conceived after union in love with Darkness."""
    },
    {
        "id": "metamorphoses",
        "name": "Ovid's Metamorphoses",
        "text": """Before the sea and lands began to be, before the sky had mantled everything, Nature displayed a single face, which they called Chaos: a raw and undivided mass, nothing but weight, lifeless, whose components were heaped together, where seeds of things had been rudely lumped, of things in strife, things jostled by things."""
    }
]

---

## Step 1: Define the Card Schema

Ask the AI:

> **Create a Pydantic schema called `PlayerCard` for a literary text card. It should have: `title` (str), `source` (str), `era` (str), `themes` (list of str), `key_images` (list of str), `mood` (str), `flavor_quote` (str), `power_rating` (int from 1-10), and `art_prompt` (str - a description for generating card art).**

Paste the schema:

In [ ]:
# Step 1: Paste your PlayerCard Pydantic schema here
# (Define a class with: title, source, era, themes, key_images, mood, flavor_quote, power_rating, art_prompt)



---

## Step 2: Create the Functions

We need three functions:

**1. `extract_card(name, text)`** — Extract structured data from a passage

Ask the AI:
> **Create a function that uses `client.beta.chat.completions.parse()` with the `PlayerCard` schema to extract card data from a passage.**

In [ ]:
# Step 2a: Paste your extract_card(name, text) function here
# (Uses client.beta.chat.completions.parse() with PlayerCard schema)



**2. `generate_card_art(art_prompt, card_id)`** — Generate an image with DALL-E and save it

Ask the AI:
> **Create a function that uses `client.images.generate()` with DALL-E 3 to generate an image from a prompt. It should download the image, save it to `IMAGES_DIR` with the card_id as filename, and return the web path `/cards/{card_id}.png`.**

In [ ]:
# Step 2b: Paste your generate_card_art(art_prompt, card_id) function here
# (Uses client.images.generate() with DALL-E 3, saves image to IMAGES_DIR)



**3. `process_passage(passage)`** — Combine extraction and image generation

Ask the AI:
> **Create a function that takes a passage dict (with `id`, `name`, `text` keys), extracts the card data, generates the art, and returns a complete dict with all card data plus `id`, `image_path`, and `original_text` fields.**

In [ ]:
# Step 2c: Paste your process_passage(passage) function here
# (Combines extract_card + generate_card_art, returns complete card dict)



---

## Step 3: Test on One Passage

In [ ]:
# Test on Genesis
print("Processing Genesis...")
test_card = process_passage(passages[0])
print(f"  ✓ Done!")
print()
print("Card data:")
print(json.dumps(test_card, indent=2))

In [ ]:
# Display the generated image
print(f"Title: {test_card['title']}")
print(f"Image saved to: {test_card['image_path']}")
display(Image(filename=str(IMAGES_DIR / f"{passages[0]['id']}.png"), width=400))

---

## Step 4: Process All Passages

Ask the AI:
> **Loop through the `passages` list and call `process_passage()` on each one. Store the results in a list called `cards`. Skip re-processing Genesis if we already have `test_card`.**

In [ ]:
# Step 4: Paste your loop code here
# (Loop through passages list, call process_passage() on each, store results in cards list)



---

## Step 5: Display All Cards

Ask the AI:
> **Create a `display_card(card)` function that shows the card's image and key info (title, source, era, mood, power rating, themes, quote).**

In [ ]:
# Step 5: Paste your display_card(card) function here
# (Shows card image and key info: title, source, era, mood, power rating, themes, quote)



In [ ]:
# Step 5b: Display all cards by looping through the cards list



---

## Step 6: Save Data for Next.js App

Save the card data as JSON so it can be used by the Next.js interface app.

In [ ]:
# Save cards data to JSON
output_file = DATA_DIR / "cards.json"

with open(output_file, "w") as f:
    json.dump(cards, f, indent=2)

print(f"✓ Saved card data to: {output_file}")

In [ ]:
# Verify what we saved
print("Files created:")
print()
print(f"Images ({IMAGES_DIR}):")
for f in IMAGES_DIR.glob("*.png"):
    print(f"  {f.name}")
print()
print(f"Data ({DATA_DIR}):")
for f in DATA_DIR.glob("*.json"):
    print(f"  {f.name}")

---

## Step 7: Compare the Cards

In [ ]:
# Step 7a: Compare power ratings across all cards
# (Print each card's title and power_rating)



In [ ]:
# Step 7b: Find common themes across all cards
# (Collect all themes and find which ones appear in multiple cards)



---

## What You Built

You just:
1. Defined a **schema** for what you wanted
2. Created **functions** to extract data and generate art
3. **Looped** through 3 passages automatically
4. Got **structured data** with generated images
5. **Saved everything** to Google Drive

Your files are now in your Google Drive:
- **Images:** `MyDrive/complit-126x/cards/*.png`
- **Data:** `MyDrive/complit-126x/data/cards.json`

---

## The Three Powers of Code

| Power | What it means | You used it when... |
|-------|--------------|--------------------|
| **SCALE** | Process many things automatically | You looped through 3 passages |
| **CHAINING** | Connect steps into workflows | You extracted → generated → saved |
| **CONTROL** | Get structured data, not paragraphs | You defined a schema and got organized cards |

These three powers are why coding with AI beats vanilla ChatGPT for serious work.

---

## What's Next?

Your card data and images are now in the Next.js app folder. Next steps:

1. **Build the UI** — Vibecode a Next.js page that displays the cards
2. **Add more passages** — Run this notebook with more texts
3. **Customize the schema** — Add fields like author, genre, literary devices
4. **Create a gallery** — Build a searchable, filterable card collection

You now have the tools. Go build something.